# 2 · Hyperparameter search

Finds good settings for the three model families on the annotated series 1.

* The window features are computed **once** per sausage (`ml_core.training.load_windows`). Windowing is
  stateless, so this gives the same result as windowing inside every fold, but much faster.
* Successive halving grid search (`HalvingGridSearchCV`) with **leave-one-sausage-out** cross-validation,
  scored by **balanced accuracy**, because the phases have very different lengths.
* The winners are written to `data/models/hyperparameters.json`. Notebook 3 reads them from there.
* Search space: `PARAM_GRIDS` in `ml_core/training.py`. `QUICK = True` runs tiny grids, for testing the notebook only.

Runtime on the full data: tens of minutes on CPU. If RAPIDS is installed (GPU), the next cell switches it on.

In [ ]:
# Optional GPU acceleration. Must run before scikit-learn is imported.
try:
    import cuml
    cuml.accel.install()
    GPU = True
except Exception as e:  # not installed or no GPU: plain scikit-learn on CPU
    GPU = False
print("RAPIDS cuML acceleration:", GPU)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import pandas as pd
from ml_core import data
from ml_core.training import (MODEL_NAMES, PARAM_GRIDS, QUICK_GRIDS, load_windows,
                              save_hyperparameters, search_hyperparameters)
pd.set_option("display.max_colwidth", 200)

In [ ]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = 10, 2
QUICK = False
N_JOBS = 1 if GPU else -1

In [ ]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE)
print(f"{len(windows.columns)} features per window")
windows.summary()

In [ ]:
result = search_hyperparameters(windows, grids=QUICK_GRIDS if QUICK else PARAM_GRIDS, n_jobs=N_JOBS)

## Best candidates per model (last halving round)

In [ ]:
for key, r in result.items():
    print(f"{MODEL_NAMES[key]}: best balanced accuracy {r['score']:.3f}")
    display(r["results"].head(5))

In [ ]:
path = save_hyperparameters(result, windows, MODELS_DIR)
print("Saved", path)